In [39]:
from pathlib import Path
import duckdb
import requests
from tqdm import tqdm
import pandas as pd
import numpy as np

In [40]:
DATA_DIR = Path("data")
CATEGORY = "Kindle_Store"
BASE_URL = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw"
REVIEWS_URL = f"{BASE_URL}/review_categories/{CATEGORY}.jsonl.gz"
META_URL    = f"{BASE_URL}/meta_categories/meta_{CATEGORY}.jsonl.gz"
REVIEWS_FILE = DATA_DIR / f"{CATEGORY}.jsonl.gz"
META_FILE    = DATA_DIR / f"meta_{CATEGORY}.jsonl.gz"
OUTPUT_FILE  = DATA_DIR / f"{CATEGORY}_merged.parquet"

In [41]:
c2 = duckdb.connect()

In [42]:
head_reviews = c2.execute(f"SELECT * FROM read_json_auto('{REVIEWS_URL}') LIMIT 5").df()
head_reviews

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5.0,excellent writer reminds me of Clive Cussler,GRUMLEY is on par with Clive Cussler and his D...,[],B00LXRJICK,B00LXRJICK,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1427541413000,0,False
1,3.0,Alright book,The book Fade was not my favorite but was a go...,[],B073DFP8VC,B073DFP8VC,AHGTHCERTEZUXNBLJ5SWHK2CDLXA,1504226946142,0,True
2,5.0,Hats off to Fern Michaels for all her great ac...,I have been a fan of this author for many year...,[],B07QVH25KX,B07QVH25KX,AHFY2QSS6PK5MHSYZFI6TXUYNPLQ,1644883955777,0,True
3,5.0,Great read,This book is more than just about a dog and a ...,[],B004Y1NWQU,B004Y1NWQU,AHFY2QSS6PK5MHSYZFI6TXUYNPLQ,1363027885000,0,False
4,5.0,Add to legend f Arthur,Good twist on the ledgen of King<br />Arthur !...,[],B08M993CNC,B08M993CNC,AFWHJ6O3PV4JC7PVOJH6CPULO2KQ,1637557512064,0,True


In [43]:
head_meta = c2.execute(f"SELECT * FROM read_json_auto('{META_URL}') LIMIT 5").df()
head_meta

,main_category,title,subtitle,author,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together
0,Buy a Kindle,The Palace (Chateau Book 4),Kindle Edition,{'avatar': 'https://m.media-amazon.com/images/...,4.7,970,[A line was drawn in the sand.I made my choice...,[],0.00,[{'large': 'https://m.media-amazon.com/images/...,[],Penelope Sky (Author) Format: Kindle Edition,"[Kindle Store, Kindle eBooks, Romance]","{'Publisher': 'Hartwick Publishing (May 25, 20...",B08XPZPFY4,None
1,Buy a Kindle,Microsoft PowerPoint 2016 2013 2010 2007 Tips ...,[Print Replica] Kindle Edition,{'avatar': 'https://m.media-amazon.com/images/...,4.3,35,"[Paperback versions are also available, includ...","[From the Author, Amelia Griggs is an Instruct...",0.00,[{'large': 'https://m.media-amazon.com/images/...,[],Amelia Griggs (Author) Format: Kindle Edition,"[Kindle Store, Kindle eBooks, Computers & Tech...","{'Publisher': None, 'Publication date': 'June ...",B07DH1LF1K,None
2,Buy a Kindle,Ill Wind (Anna Pigeon Mysteries Book 3),Kindle Edition,{'avatar': 'https://m.media-amazon.com/images/...,4.4,1628,"[Lately, visitors to Mesa Verde have been brin...","[From Publishers Weekly, Barr lands another su...",7.99,[{'large': 'https://m.media-amazon.com/images/...,[],Nevada Barr (Author) Format: Kindle Edition,"[Kindle Store, Kindle eBooks, Mystery, Thrille...",{'Publisher': 'Berkley; Reissue edition (March...,B0022Q8CTQ,None
3,Buy a Kindle,30 Healthy Easy Quick Lentil Recipes (Brad Arm...,Kindle Edition,{'avatar': 'https://m.media-amazon.com/images/...,3.8,47,"[If improved health you are seeking, look no f...",[],0.00,[{'large': 'https://m.media-amazon.com/images/...,[],Brad Armstrong (Author) Format: Kindle Edition,"[Kindle Store, Kindle eBooks, Health, Fitness ...","{'Publisher': 'Brad Armstrong (March 10, 2013)...",B00BS56MLC,None
4,Buy a Kindle,The Road Home,Kindle Edition,{'avatar': 'https://m.media-amazon.com/images/...,4.5,475,"[In one of Jim Harrison’s greatest works, five...","[Review, ""A graceful novel...To read this book...",10.44,[{'large': 'https://m.media-amazon.com/images/...,[],Jim Harrison (Author) Format: Kindle Edition,"[Kindle Store, Kindle eBooks, Literature & Fic...",{'Publisher': 'Atlantic Monthly Press (Decembe...,B00155EZRS,None


In [44]:
c2.execute(f"""
    COPY (
        SELECT * FROM read_json_auto('{REVIEWS_URL}')
        LIMIT 20000
    )
    TO '../data/raw/reviews_raw.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

In [45]:
c2.execute(f"""
      COPY (SELECT * FROM read_json_auto('{META_URL}') LIMIT 20000)
      TO '../data/raw/meta_raw.parquet'
      (FORMAT PARQUET, COMPRESSION ZSTD)
  """)

In [46]:
reviews_df = c2.execute(f"SELECT * FROM read_parquet('../data/raw/reviews_raw.parquet')").df()
meta_df = c2.execute(f"SELECT * FROM read_parquet('../data/raw/meta_raw.parquet')").df()

In [47]:
reviews_df.describe(), reviews_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   rating             20000 non-null  float64
 1   title              20000 non-null  str    
 2   text               20000 non-null  str    
 3   images             20000 non-null  object 
 4   asin               20000 non-null  str    
 5   parent_asin        20000 non-null  str    
 6   user_id            20000 non-null  str    
 7   timestamp          20000 non-null  int64  
 8   helpful_vote       20000 non-null  int64  
 9   verified_purchase  20000 non-null  bool   
dtypes: bool(1), float64(1), int64(2), object(1), str(5)
memory usage: 16.7+ MB


(             rating     timestamp  helpful_vote
 count  20000.000000  2.000000e+04   20000.00000
 mean       4.314750  1.520103e+12       1.31520
 std        0.987386  9.431514e+10      10.29958
 min        1.000000  1.151076e+12       0.00000
 25%        4.000000  1.445269e+12       0.00000
 50%        5.000000  1.529546e+12       0.00000
 75%        5.000000  1.598719e+12       1.00000
 max        5.000000  1.679116e+12     888.00000,
 None)

In [48]:

meta_df.describe(), meta_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   main_category    20000 non-null  str    
 1   title            20000 non-null  str    
 2   subtitle         19644 non-null  str    
 3   author           16898 non-null  object 
 4   average_rating   20000 non-null  float64
 5   rating_number    20000 non-null  int64  
 6   features         20000 non-null  object 
 7   description      20000 non-null  object 
 8   price            18894 non-null  float64
 9   images           20000 non-null  object 
 10  videos           20000 non-null  object 
 11  store            20000 non-null  str    
 12  categories       20000 non-null  object 
 13  details          20000 non-null  object 
 14  parent_asin      20000 non-null  str    
 15  bought_together  0 non-null      object 
dtypes: float64(2), int64(1), object(8), str(5)
memory usage: 5.2+ MB


(       average_rating  rating_number         price
 count    20000.000000   20000.000000  18894.000000
 mean         4.315080     391.015450      3.892047
 std          0.509733    2181.193826      9.856055
 min          1.000000       1.000000      0.000000
 25%          4.100000      12.000000      0.000000
 50%          4.400000      44.000000      0.990000
 75%          4.600000     194.250000      4.990000
 max          5.000000  138549.000000    982.990000,
 None)

In [ ]:

#Query only the columns neded/selected, and use Inner join. We wont use LEFT to avoid many missing values.

c2.execute("""
    COPY (
         SELECT r.parent_asin, r.rating, r.title AS review_title, r.text AS review_text, r.verified_purchase,
            m.title AS product_title, m.average_rating, m.main_category, m.description, m.features
        FROM read_parquet('../data/raw/reviews_raw.parquet') r
        INNER JOIN read_parquet('../data/raw/meta_raw.parquet') m USING (parent_asin)
    )
    TO '../data/processed/merged.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

In [50]:
merged_df = c2.execute(f"SELECT * FROM read_parquet('../data/processed/merged.parquet')").df()

In [51]:
merged_df.head(5)

,parent_asin,rating,review_title,review_text,verified_purchase,product_title,average_rating,main_category,description,features
0,B091N666DM,5.0,Great series,"Loved this addition to series, lots of good ti...",True,Missy Goes to Annapolis (Missy the Werecat Boo...,4.6,Buy a Kindle,"[From the Author, *** New Adult book best suit...","[Missy Goes to Annapolis: Book IX, After her s..."
1,B087ZXSPGJ,5.0,Another brilliant KH novel,This wasn’t a happy read. It is a hard read bu...,True,The Four Winds: A Novel,4.5,Buy a Kindle,"[Amazon.com Review, An Amazon Best Book of Feb...","[""The Bestselling Hardcover Novel of the Year...."
2,B06XQ85LY5,3.0,I like the plot so I'll keep reading,The writing is lacking but the characters are ...,False,Obsidian Magic (Legacy Series Book 2),4.4,Buy a Kindle,"[Review, ""This series gets, better, and, bette...",[The only thing standing between me and death ...
3,B0092SIL52,5.0,Five Stars,Are you a fan of Heart? This is an engaging read.,True,Kicking & Dreaming (Enhanced Edition) v2: A St...,4.5,Buy a Kindle,"[Review, “Righteously entertaining….[it] shows...","[In the enhanced e-book edition of, Kicking an..."
4,B00LD1RZ3A,5.0,One of my favorite authors and I loved it so m...,One of my favorite authors and I loved it so m...,True,Less Than Hero,4.4,Buy a Kindle,"[Unknown, ""Wickedly sharp and wildly entertain...",[With the razor-sharp satire that earned him r...


In [52]:
merged_df.info(), merged_df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 305 entries, 0 to 304
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   parent_asin        305 non-null    str    
 1   rating             305 non-null    float64
 2   review_title       305 non-null    str    
 3   review_text        305 non-null    str    
 4   verified_purchase  305 non-null    bool   
 5   product_title      305 non-null    str    
 6   average_rating     305 non-null    float64
 7   main_category      305 non-null    str    
 8   description        305 non-null    object 
 9   features           305 non-null    object 
dtypes: bool(1), float64(2), object(2), str(5)
memory usage: 250.8+ KB


(None,
            rating  average_rating
 count  305.000000      305.000000
 mean     4.390164        4.362623
 std      0.915097        0.271048
 min      1.000000        3.400000
 25%      4.000000        4.200000
 50%      5.000000        4.400000
 75%      5.000000        4.600000
 max      5.000000        5.000000)

In [53]:
merged_df.isna().mean().sort_values(ascending=False)

parent_asin          0.0
rating               0.0
review_title         0.0
review_text          0.0
verified_purchase    0.0
product_title        0.0
average_rating       0.0
main_category        0.0
description          0.0
features             0.0
dtype: float64

In [54]:

#Funtion to convert list of strings in 1 single string. For each list of features and descriptions.
def to_text(x):
    if x is pd.NA:
        return ""
    if isinstance(x, np.ndarray):
        return " ".join(map(str, x))
    return str(x)

merged_df["features"] = merged_df["features"].apply(to_text)
merged_df["description"] = merged_df["description"].apply(to_text)

In [55]:
merged_df.head(3)

,parent_asin,rating,review_title,review_text,verified_purchase,product_title,average_rating,main_category,description,features
0,B091N666DM,5.0,Great series,"Loved this addition to series, lots of good ti...",True,Missy Goes to Annapolis (Missy the Werecat Boo...,4.6,Buy a Kindle,From the Author *** New Adult book best suited...,Missy Goes to Annapolis: Book IX After her sec...
1,B087ZXSPGJ,5.0,Another brilliant KH novel,This wasn’t a happy read. It is a hard read bu...,True,The Four Winds: A Novel,4.5,Buy a Kindle,Amazon.com Review An Amazon Best Book of Febru...,"""The Bestselling Hardcover Novel of the Year.""..."
2,B06XQ85LY5,3.0,I like the plot so I'll keep reading,The writing is lacking but the characters are ...,False,Obsidian Magic (Legacy Series Book 2),4.4,Buy a Kindle,"Review ""This series gets better and better wit...",The only thing standing between me and death i...


In [56]:
#From the selected columns, lets investigate which columns are meaningful
merged_df.nunique()

parent_asin          287
rating                 5
review_title         292
review_text          305
verified_purchase      2
product_title        287
average_rating        16
main_category          2
description          175
features             285
dtype: int64

In [ ]:
# Add to document column the columns that will provide the most context. 
merged_df["document"] = (
    "Product: " + merged_df["product_title"].fillna("") + ". "
    + "Description: " + merged_df["description"].fillna("") + ". "
    + "Features: " + merged_df["features"].fillna("") + ". "
    + "Review Title: " + merged_df["review_title"].fillna("") + ". "
    + "Review: " + merged_df["review_text"].fillna("")
)

In [63]:
merged_df['document'][1]

'Product: The Four Winds: A Novel. Description: Amazon.com Review An Amazon Best Book of February 2021: No, the grit stinging your eyes and getting stuck in your teeth isn’t real, that’s just how evocative Kristin Hannah’s descriptions of the Dust Bowl storms are. But unlikely heroine, Elsa Martinelli, will lodge herself into your heart. The Four Winds is a reminder, when we so urgently need it, of the resiliency not only of the human spirit, but of this country as well. Hannah\'s latest reads like a classic. –Erin Kodicek, Amazon Book Review Review "The Bestselling Hardcover Novel of the Year."--Publishers Weekly Book of the Month Club\'s Best Book of 2021 Selected for The Texas Library Association\'s 2022 Lariat Adult Fiction Reading List " The Four Winds seems eerily prescient in 2021 . . . Its message is galvanizing and hopeful: We are a nation of scrappy survivors. We’ve been in dire straits before; we will be again. Hold your people close.” ― The New York Times "A spectacular tou